# 03 — Reaction-Diffusion Simulation

2D multicellular EMT on a 50×50 grid.
- Operator splitting: RK4 (reaction) + explicit Euler (diffusion)
- Neumann (zero-flux) BCs
- Numba JIT + prange parallelism

Followed by a full RD bifurcation sweep.
Saves `results/data/frames_I50k.npy` — loaded by notebook 04.

## Imports & parameters

In [ ]:
import numpy as np
import math
import time
import matplotlib.pyplot as plt
from numba import njit, prange
from google.colab import output
output.enable_custom_widget_manager()

GRID_SIZE  = 50
STATE_SIZE = 7
T_END      = 500.0
DT         = 0.05
STEPS      = int(T_END / DT)
SAVE_EVERY = 20
D_COEFF    = np.array([0.5, 0.8, 0.6, 0.4, 0.4, 0.5, 0.0], dtype=np.float64)

print(f"Grid: {GRID_SIZE}×{GRID_SIZE} | Steps: {STEPS} | Frames: {STEPS//SAVE_EVERY+1}")


## Molecular parameters

In [ ]:
L_ARR       = np.array([1.0, 0.6, 0.3, 0.1, 0.05, 0.05, 0.05], dtype=np.float64)
GAMMA_MRNA  = np.array([0.0, 0.04, 0.2, 1.0, 1.0,  1.0,  1.0],  dtype=np.float64)
GAMMA_MIRNA = np.array([0.0, 0.005, 0.05, 0.5, 0.5, 0.5, 0.5],  dtype=np.float64)

COMB_6 = np.array([math.comb(6, i) for i in range(7)], dtype=np.float64)
COMB_2 = np.array([math.comb(2, i) for i in range(3)], dtype=np.float64)
IDX_7  = np.arange(7, dtype=np.float64)
IDX_3  = np.arange(3, dtype=np.float64)

T_MIR200=10e3; T_MIR34=10e3; T_MIR34_SNAIL=300e3; T_MSNAIL_SNAIL=200e3
T_MIR34_ZEB=600e3; T_MSNAIL_I=50e3; T_MIR200_ZEB=220e3; T_MIR200_SNAIL=180e3
T_MZEB_ZEB=25e3; T_MZEB_SNAIL=180e3
N_MIR34_SNAIL=1; N_MIR34_ZEB=1; N_MSNAIL_SNAIL=1; N_MSNAIL_I=1
N_MIR200_ZEB=3;  N_MIR200_SNAIL=2; N_MZEB_ZEB=2; N_MZEB_SNAIL=2
L_MIR34_SNAIL=0.1; L_MSNAIL_SNAIL=0.1; L_MIR34_ZEB=0.2; L_MSNAIL_I=10.0
L_MIR200_ZEB=0.1;  L_MIR200_SNAIL=0.1; L_MZEB_ZEB=7.5;  L_MZEB_SNAIL=10.0
G_MIR200=2.1e3; G_MZEB=11.0; G_ZEB=0.1e3
G_MIR34=1.35e3; G_MSNAIL=90.0; G_SNAIL=0.1e3
K_MIR200=0.05; K_MZEB=0.5; K_ZEB=0.1
K_MIR34=0.05;  K_MSNAIL=0.5; K_SNAIL=0.125

print("Parameters set.")


## JIT kernels

In [ ]:
@njit(fastmath=True)
def hill(x, threshold, n, leak):
    base = 1.0 / (1.0 + (x / threshold) ** n)
    return base + leak * (1.0 - base)

@njit(fastmath=True)
def mir200_terms(state):
    r = state[0] / T_MIR200
    denom = (1.0 + r) ** 6
    fac = np.power(r, IDX_7) / denom
    return (np.sum(GAMMA_MIRNA * COMB_6 * IDX_7 * fac),
            np.sum(GAMMA_MRNA  * COMB_6 * fac),
            np.sum(L_ARR       * COMB_6 * fac))

@njit(fastmath=True)
def mir34_terms(state):
    r = state[5] / T_MIR34
    denom = (1.0 + r) ** 2
    fac = np.power(r, IDX_3) / denom
    return (np.sum(GAMMA_MIRNA[:3] * COMB_2 * IDX_3 * fac),
            np.sum(GAMMA_MRNA[:3]  * COMB_2 * fac),
            np.sum(L_ARR[:3]       * COMB_2 * fac))

@njit(fastmath=True)
def cell_rhs(state):
    d = np.zeros(STATE_SIZE, dtype=np.float64)
    m200 = mir200_terms(state)
    m34  = mir34_terms(state)
    h_mir200_zeb  = hill(state[2], T_MIR200_ZEB,   N_MIR200_ZEB,   L_MIR200_ZEB)
    h_mir200_sna  = hill(state[3], T_MIR200_SNAIL, N_MIR200_SNAIL, L_MIR200_SNAIL)
    h_mzeb_zeb    = hill(state[2], T_MZEB_ZEB,     N_MZEB_ZEB,     L_MZEB_ZEB)
    h_mzeb_sna    = hill(state[3], T_MZEB_SNAIL,   N_MZEB_SNAIL,   L_MZEB_SNAIL)
    h_mir34_sna   = hill(state[3], T_MIR34_SNAIL,  N_MIR34_SNAIL,  L_MIR34_SNAIL)
    h_mir34_zeb   = hill(state[2], T_MIR34_ZEB,    N_MIR34_ZEB,    L_MIR34_ZEB)
    h_msna_sna    = hill(state[3], T_MSNAIL_SNAIL, N_MSNAIL_SNAIL, L_MSNAIL_SNAIL)
    h_msna_i      = hill(state[6], T_MSNAIL_I,     N_MSNAIL_I,     L_MSNAIL_I)
    d[0] = G_MIR200*h_mir200_zeb*h_mir200_sna - state[1]*m200[0] - K_MIR200*state[0]
    d[1] = G_MZEB  *h_mzeb_zeb  *h_mzeb_sna   - state[1]*m200[1] - K_MZEB  *state[1]
    d[2] = G_ZEB   *state[1]    *m200[2]       - K_ZEB   *state[2]
    d[3] = G_SNAIL *state[4]    *m34[2]        - K_SNAIL *state[3]
    d[4] = G_MSNAIL*h_msna_i    *h_msna_sna    - state[4]*m34[1]  - K_MSNAIL*state[4]
    d[5] = G_MIR34 *h_mir34_zeb *h_mir34_sna   - state[4]*m34[0]  - K_MIR34 *state[5]
    d[6] = 0.0
    return d

@njit(fastmath=True)
def rk4_step(state, dt):
    k1 = cell_rhs(state)
    k2 = cell_rhs(state + 0.5*dt*k1)
    k3 = cell_rhs(state + 0.5*dt*k2)
    k4 = cell_rhs(state + dt*k3)
    return state + (dt/6.0)*(k1 + 2.0*k2 + 2.0*k3 + k4)

@njit(parallel=True, fastmath=True)
def laplacian_neumann(grid, lap):
    nx, ny, ns = grid.shape
    for i in prange(1, nx-1):
        for j in range(1, ny-1):
            for k in range(ns):
                lap[i,j,k] = (grid[i+1,j,k]+grid[i-1,j,k]+
                               grid[i,j+1,k]+grid[i,j-1,k]-4.0*grid[i,j,k])
    for k in prange(ns):
        for j in range(1, ny-1):
            lap[0,j,k]    = 2.0*grid[1,j,k]+grid[0,j+1,k]+grid[0,j-1,k]-4.0*grid[0,j,k]
            lap[nx-1,j,k] = 2.0*grid[nx-2,j,k]+grid[nx-1,j+1,k]+grid[nx-1,j-1,k]-4.0*grid[nx-1,j,k]
        for i in range(1, nx-1):
            lap[i,0,k]    = grid[i+1,0,k]+grid[i-1,0,k]+2.0*grid[i,1,k]-4.0*grid[i,0,k]
            lap[i,ny-1,k] = grid[i+1,ny-1,k]+grid[i-1,ny-1,k]+2.0*grid[i,ny-2,k]-4.0*grid[i,ny-1,k]
        lap[0,0,k]       = 2.0*grid[1,0,k]   +2.0*grid[0,1,k]   -4.0*grid[0,0,k]
        lap[nx-1,0,k]    = 2.0*grid[nx-2,0,k]+2.0*grid[nx-1,1,k]-4.0*grid[nx-1,0,k]
        lap[0,ny-1,k]    = 2.0*grid[1,ny-1,k]+2.0*grid[0,ny-2,k]-4.0*grid[0,ny-1,k]
        lap[nx-1,ny-1,k] = 2.0*grid[nx-2,ny-1,k]+2.0*grid[nx-1,ny-2,k]-4.0*grid[nx-1,ny-1,k]

@njit(parallel=True, fastmath=True)
def simulate(grid, dt, steps, D, save_every):
    nx, ny, ns = grid.shape
    lap = np.zeros_like(grid); tmp = np.zeros_like(grid)
    n_frames = steps//save_every + 1
    frames = np.zeros((n_frames, nx, ny, ns), dtype=np.float64)
    mass   = np.zeros(n_frames, dtype=np.float64)
    frames[0] = grid.copy(); mass[0] = np.sum(grid); f_idx = 1
    for t in range(1, steps+1):
        laplacian_neumann(grid, lap)
        for i in prange(nx):
            for j in range(ny):
                after_diff = np.empty(ns, dtype=np.float64)
                for k in range(ns):
                    after_diff[k] = grid[i,j,k] + dt*D[k]*lap[i,j,k]
                after_rxn = rk4_step(after_diff, dt)
                for k in range(ns):
                    tmp[i,j,k] = after_rxn[k] if after_rxn[k] > 0.0 else 0.0
        grid[:,:,:] = tmp[:,:,:]
        if t % save_every == 0:
            frames[f_idx] = grid.copy(); mass[f_idx] = np.sum(grid); f_idx += 1
    return frames, mass

print("JIT kernels defined.")


## Numba warm-up (~20–40 s)

In [ ]:
print("Compiling...")
_g = np.ones((4, 4, STATE_SIZE), dtype=np.float64) * 1e3
_l = np.zeros_like(_g)
laplacian_neumann(_g, _l)
_ = cell_rhs(_g[0,0,:]); _ = rk4_step(_g[0,0,:], 0.01)
simulate(_g, 0.01, 4, D_COEFF, 2)
print("Compilation done.")


## Grid initialisation

Central 13×13 patch → M-state | Background → E-state | k=6 (signal I) → uniform

In [ ]:
np.random.seed(42)
grid_init = np.zeros((GRID_SIZE, GRID_SIZE, STATE_SIZE), dtype=np.float64)
center = GRID_SIZE // 2; half_patch = 6
HIGH_MIN=110e3; HIGH_MAX=120e3; LOW_MIN=20e3; LOW_MAX=50e3; I_SIGNAL=50e3

for k in range(STATE_SIZE - 1):
    grid_init[:, :, k] = np.random.uniform(LOW_MIN, LOW_MAX, (GRID_SIZE, GRID_SIZE))
    grid_init[center-half_patch:center+half_patch+1,
              center-half_patch:center+half_patch+1, k] =         np.random.uniform(HIGH_MIN, HIGH_MAX, (2*half_patch+1, 2*half_patch+1))

grid_init[:, :, 6] = I_SIGNAL
grid_init[:, :, :6] += np.random.rand(GRID_SIZE, GRID_SIZE, 6) * 0.5
print("Grid initialised.")


## Run simulation (~8–15 min)

In [ ]:
import os
os.makedirs('../results/data', exist_ok=True)

print(f"Running: {GRID_SIZE}×{GRID_SIZE} | Steps={STEPS} | DT={DT}")
t0 = time.time()
frames, mass = simulate(grid_init, DT, STEPS, D_COEFF, SAVE_EVERY)
print(f"Done in {time.time()-t0:.2f}s")
print(f"Initial mass: {mass[0]:.4e} | Final mass: {mass[-1]:.4e} | NaN: {np.isnan(frames).any()}")

np.save('../results/data/frames_I50k.npy', frames)
np.save('../results/data/mass_I50k.npy', mass)
print("Frames saved.")


## Interactive heatmap widget

In [ ]:
from ipywidgets import Play, IntSlider, jslink, ToggleButtons, interact
from IPython.display import display

species_map = {"miR200":0, "mZEB":1, "ZEB":2, "SNAIL":3, "mSNAIL":4, "miR34":5}
plt.rcParams['figure.facecolor'] = '#ffcba4'

def plot_at_time(idx, species):
    k    = species_map[species]
    data = frames[idx, :, :, k]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    im = axes[0].imshow(data, cmap='viridis', interpolation='bilinear',
                        origin='lower', vmin=data.min(), vmax=data.max())
    fig.colorbar(im, ax=axes[0], label=f"{species} concentration")
    axes[0].set_title(f"{species}  |  t = {idx*SAVE_EVERY*DT:.2f}  |  mass = {mass[idx]:.2e}")
    axes[0].set_xlabel("x (cell index)"); axes[0].set_ylabel("y (cell index)")
    t_axis = np.arange(frames.shape[0]) * SAVE_EVERY * DT
    axes[1].plot(t_axis, mass, color='steelblue', lw=1.6)
    axes[1].axvline(idx*SAVE_EVERY*DT, color='tomato', lw=1.4, ls='--')
    axes[1].scatter([idx*SAVE_EVERY*DT], [mass[idx]], color='tomato', zorder=5, s=40)
    axes[1].set_xlabel("Time"); axes[1].set_ylabel("Total mass")
    axes[1].set_title("Mass evolution"); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

species_toggle = ToggleButtons(options=list(species_map.keys()), value="ZEB",
                                description="Species:", button_style="info")
play   = Play(value=0, min=0, max=frames.shape[0]-1, step=1, interval=100, description="Play")
slider = IntSlider(value=0, min=0, max=frames.shape[0]-1, step=1, description='Frame')
jslink((play, 'value'), (slider, 'value'))
interact(plot_at_time, idx=slider, species=species_toggle)
jslink((play, 'value'), (slider, 'value'))
display(play)


---
# RD Bifurcation

Full 2D quasi-static sweep — entire 50×50 grid evolved to steady state per I value.

> Runtime: ~10–25 min for 50 I values per arm.

In [ ]:
ROW, COL = GRID_SIZE//2, GRID_SIZE//2
N_I=50; I_MIN=20e3; I_MAX=120e3; I_MID_INIT=65e3
T_END_SS=500.0; DT_SS=0.05; STEPS_SS=int(T_END_SS/DT_SS)

def build_grid(I_val, seed=42):
    np.random.seed(seed)
    g = np.zeros((GRID_SIZE, GRID_SIZE, STATE_SIZE), dtype=np.float64)
    c=GRID_SIZE//2; hp=6
    for k in range(STATE_SIZE-1):
        g[:,:,k] = np.random.uniform(LOW_MIN, LOW_MAX, (GRID_SIZE, GRID_SIZE))
        g[c-hp:c+hp+1, c-hp:c+hp+1, k] =             np.random.uniform(HIGH_MIN, HIGH_MAX, (2*hp+1, 2*hp+1))
    g[:,:,6] = I_val
    g[:,:,:6] += np.random.rand(GRID_SIZE, GRID_SIZE, 6) * 0.5
    return g

def run_rd_single(grid_ic):
    f, _ = simulate(grid_ic, DT_SS, STEPS_SS, D_COEFF, STEPS_SS)
    return f[-1]

def sweep_rd(I_values, grid_ic):
    results = np.empty((len(I_values), STATE_SIZE), dtype=np.float64)
    grid = grid_ic.copy()
    for idx, I_val in enumerate(I_values):
        t0=time.time()
        grid[:,:,6] = I_val
        final_grid = run_rd_single(grid)
        results[idx] = final_grid[ROW, COL, :]
        grid = final_grid
        if idx%10==0 or idx==len(I_values)-1:
            print(f"  [{idx+1:3d}/{len(I_values)}]  I={I_val/1e3:.1f}k  "
                  f"mZEB={results[idx,1]:.2e}  ({time.time()-t0:.1f}s)")
    return results, grid

I_forward  = np.linspace(I_MIN, I_MAX, N_I)
I_backward = np.linspace(I_MAX, I_MIN, N_I)
I_mid_back = np.linspace(I_MID_INIT, I_MIN, N_I)

print(f"Forward sweep (E→M)...")
res_fwd, grid_after_fwd = sweep_rd(I_forward, build_grid(I_MIN))

print(f"\nBackward sweep (M→E)...")
res_bwd, _ = sweep_rd(I_backward, grid_after_fwd)

print(f"\nMid-backward sweep...")
grid_start_mid = run_rd_single(build_grid(I_MID_INIT))
res_mid, _ = sweep_rd(I_mid_back, grid_start_mid)


## RD Bifurcation plot

In [ ]:
mzeb_fwd=res_fwd[:,1]; snail_fwd=res_fwd[:,3]
mzeb_bwd=res_bwd[:,1]; snail_bwd=res_bwd[:,3]
mzeb_mid=res_mid[:,1]; snail_mid=res_mid[:,3]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Full RD bifurcation — cell ({ROW},{COL}) | Grid {GRID_SIZE}×{GRID_SIZE} | {N_I} I values per arm", fontsize=11)

ax1=axes[0]
ax1.plot(I_forward/1e3,  mzeb_fwd, 'b.-', ms=4, lw=0.8, label='Forward (E→M)')
ax1.plot(I_backward/1e3, mzeb_bwd, 'r.-', ms=4, lw=0.8, label='Backward (M→E)')
ax1.plot(I_mid_back/1e3, mzeb_mid, 'g.-', ms=4, lw=0.8, label='Mid-backward (hybrid)')
ax1.set_xlabel('Input signal I  (×10³)', fontsize=12); ax1.set_ylabel('mZEB  (steady state)', fontsize=12)
ax1.set_title('Bifurcation diagram', fontsize=13); ax1.legend(fontsize=9)

ax2=axes[1]
ax2.plot(snail_fwd, mzeb_fwd, 'b.-', ms=4, lw=0.8, label='Forward (E→M)')
ax2.plot(snail_bwd, mzeb_bwd, 'r.-', ms=4, lw=0.8, label='Backward (M→E)')
ax2.plot(snail_mid, mzeb_mid, 'g.-', ms=4, lw=0.8, label='Mid-backward (hybrid)')
ax2.set_xlabel('SNAIL  (steady state)', fontsize=12); ax2.set_ylabel('mZEB   (steady state)', fontsize=12)
ax2.set_title('mZEB vs SNAIL', fontsize=13); ax2.legend(fontsize=9)

plt.tight_layout()
fname = f'../results/figures/bifurcation_full_RD_cell_{ROW}_{COL}.png'
plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.show()
print(f"Saved: {fname}")
